# PRIMA dense-anchor point-to-plane articulation: `hippo_05` failure case

This notebook runs `articulate/config/dense_anchor_point_to_plane_prima.yaml` from the raw `hippo_05` scan through fresh PRIMA inference, robust joint and dense-anchor fitting, sampled point-to-plane ICP, bidirectional Chamfer polishing, confidence-weighted fusion, and the production output handler. `hippo_05` is a known failure: the scalar fitting diagnostics can improve even when the articulated surface is visually wrong. The plots therefore emphasize stage boundaries, correspondence geometry, and spatial failure signatures.

Run the notebook with `pkg/PRIMA/.venv/bin/python`. The expensive cells require CUDA, the PRIMA Stage 1 checkpoint, detector weights, and `/home/knpob/work/data/shape-corr/SMAL_r/off/hippo_05.off`. No saved fit is used as an input; the existing CLI result is consulted only for an optional final comparison.

## 1. Repository, imports, and exact Hydra preset

In [ ]:
import os
import sys
from pathlib import Path


def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "articulate/config/dense_anchor_point_to_plane_prima.yaml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the atm-dev repository root.")


PROJECT_ROOT = find_project_root(Path.cwd())
PRIMA_ROOT = PROJECT_ROOT / "pkg/PRIMA"
os.chdir(PROJECT_ROOT)
for path in (PROJECT_ROOT, PRIMA_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"Repository: {PROJECT_ROOT}")

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import torch
import trimesh
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from omegaconf import OmegaConf

from articulate.shape_refiner.dense_anchor_mhr_refiner import triangulation_covariances
from articulate.shape_refiner.dense_anchor_point_to_plane_smal_refiner import (
    DenseAnchorPointToPlanePRIMASMALShapeRefiner,
)
from articulate.shape_refiner.dense_anchor_smal_refiner import (
    closest_points_on_mesh,
    project_smal_anchor_vertices,
)

PYVISTA_BACKEND = "static"
pv.set_jupyter_backend(PYVISTA_BACKEND)
pv.global_theme.background = "white"
pv.global_theme.font.color = "black"

In [ ]:
SHAPE_NAME = "hippo_05"
RAW_DIR = Path("/home/knpob/work/data/shape-corr/SMAL_r/off")
RAW_PATH = RAW_DIR / f"{SHAPE_NAME}.off"
NOTEBOOK_OUT_DIR = PROJECT_ROOT / "output/notebook_smal_prima_dense_anchor_point_to_plane_hippo05_e2e"
CLI_REFERENCE_PATH = PROJECT_ROOT / "output/smal_r_prima_anchor_p2p_no_iou/smal" / f"{SHAPE_NAME}.obj"
DEVICE = "cuda"

overrides = [
    f"shape_dir={RAW_DIR}",
    f"out_dir={NOTEBOOK_OUT_DIR}",
    f"shape_loader.filename_filter={SHAPE_NAME}",
    f"device={DEVICE}",
]
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "articulate/config"), version_base=None):
    cfg = compose(
        config_name="dense_anchor_point_to_plane_prima",
        overrides=overrides,
    )

required = [RAW_PATH, PROJECT_ROOT / cfg.shape_estimator.checkpoint_path, PRIMA_ROOT]
missing = [str(path) for path in required if not path.exists()]
assert not missing, "Missing required assets: " + ", ".join(missing)
if str(cfg.device).startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook performs fresh PRIMA inference and optimization; run it in "
        "pkg/PRIMA/.venv on a machine with an available CUDA GPU."
    )

assert cfg.shape_refiner._target_.endswith("DenseAnchorPointToPlanePRIMASMALShapeRefiner")
assert cfg.shape_refiner.n_iters_anchors == 1000
assert cfg.shape_refiner.n_iters_icp == 500
assert cfg.shape_refiner.icp_correspondence_interval == 10
assert cfg.shape_refiner.icp_tgt_sample_num == 50000
assert cfg.shape_refiner.n_iters_surface == 100
assert cfg.shape_refiner.n_views_surface == 0
assert cfg.shape_refiner.src_to_tgt and cfg.shape_refiner.tgt_to_src
assert not cfg.shape_refiner.early_stop_chamfer_plateau
assert not cfg.shape_refiner.early_stop_validation_regression

print(OmegaConf.to_yaml(cfg, resolve=True))

In [ ]:
def as_polydata(vertices, faces):
    vertices = np.asarray(vertices)
    faces = np.asarray(faces, dtype=np.int64)
    pv_faces = np.column_stack([np.full(len(faces), 3), faces]).ravel()
    return pv.PolyData(vertices, pv_faces)


def set_parallel_view(plotter, view):
    getattr(plotter, f"view_{view}")()
    plotter.enable_parallel_projection()


def line_polydata(starts, ends):
    starts = np.asarray(starts)
    ends = np.asarray(ends)
    points = np.concatenate([starts, ends], axis=0)
    count = len(starts)
    lines = np.column_stack(
        [np.full(count, 2), np.arange(count), np.arange(count) + count]
    ).ravel()
    result = pv.PolyData(points)
    result.lines = lines
    return result


def rgba_overlay(rgb, rgba, alpha=0.45):
    rgb = np.asarray(rgb, dtype=np.float32) / 255.0
    rgba = np.asarray(rgba, dtype=np.float32)
    if rgba.max() > 1.0:
        rgba = rgba / 255.0
    mask = rgba[..., 3:4] if rgba.shape[-1] == 4 else np.ones_like(rgba[..., :1])
    return np.clip((1.0 - alpha * mask) * rgb + alpha * mask * rgba[..., :3], 0.0, 1.0)


def sampled_surface_gap(source_mesh, target_mesh, sample_count=4096, seed=0):
    points, _ = trimesh.sample.sample_surface_even(source_mesh, sample_count, seed=seed)
    _, distances, _ = closest_points_on_mesh(target_mesh, points)
    return float(np.mean(distances)), float(np.median(distances))

## 2. Notebook-only tracing around the production refiner

The subclass below does not replace any optimizer or change any return value. It delegates to the production implementation and retains only detached stage meshes, per-refresh correspondence counts, and the first/latest frozen ICP matches. This makes the point-to-plane stage inspectable without moving notebook behavior into production code.

In [ ]:
class TracingDenseAnchorPointToPlanePRIMARefiner(
    DenseAnchorPointToPlanePRIMASMALShapeRefiner
):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.stage_vertices = {}
        self.icp_boundary_vertices = []
        self.icp_boundary_chamfers = []
        self.icp_refresh_history = []
        self.icp_first_snapshot = None
        self.icp_latest_snapshot = None

    def _capture_parameters(self, name, results, shape_prior, parameters):
        betas, global_orient, pose = parameters
        rotation, translation, scale = self.stack_alignment_params(
            results, dtype=betas.dtype
        )
        with torch.no_grad():
            vertices, _ = shape_prior.generate(
                betas, global_orient, pose, rotation, scale, translation
            )
        self.stage_vertices[name] = vertices.detach().cpu().numpy()

    def _sparse_anchor_optim(self, results, shape_prior, betas, global_orient, pose):
        self._capture_parameters(
            "post_joint", results, shape_prior, (betas, global_orient, pose)
        )
        optimized = super()._sparse_anchor_optim(
            results, shape_prior, betas, global_orient, pose
        )
        self._capture_parameters("post_sparse", results, shape_prior, optimized)
        return optimized

    def _validation_chamfer_for_parameters(
        self, generate_vertices, model_indices, raw_points, probes, faces, canonical_vertices
    ):
        with torch.no_grad():
            vertices = generate_vertices().detach().cpu().numpy()
        value = super()._validation_chamfer_for_parameters(
            generate_vertices, model_indices, raw_points, probes, faces, canonical_vertices
        )
        self.icp_boundary_vertices.append(vertices)
        self.icp_boundary_chamfers.append(float(value))
        return value

    def _refresh_icp_correspondences(
        self, vertices, faces, source_indices, target_points, target_normals, tree, maximum_distance
    ):
        correspondences, used_fallback = super()._refresh_icp_correspondences(
            vertices, faces, source_indices, target_points, target_normals, tree, maximum_distance
        )
        snapshot = {
            "source": vertices[:, source_indices].detach().cpu().numpy(),
            "targets": correspondences.targets.detach().cpu().numpy(),
            "normals": correspondences.normals.detach().cpu().numpy(),
            "valid": correspondences.valid.detach().cpu().numpy(),
            "accepted_per_view": correspondences.accepted_per_view.copy(),
            "used_fallback": bool(used_fallback),
        }
        self.icp_refresh_history.append(snapshot["accepted_per_view"])
        if self.icp_first_snapshot is None:
            self.icp_first_snapshot = snapshot
        self.icp_latest_snapshot = snapshot
        return correspondences, used_fallback

    def _dense_optim(self, results, mesh, shape_prior, betas, global_orient, pose):
        optimized = super()._dense_optim(
            results, mesh, shape_prior, betas, global_orient, pose
        )
        self._capture_parameters("post_chamfer", results, shape_prior, optimized)
        return optimized

## 3. Instantiate and initialize in production order

In [ ]:
shape_loader = instantiate(cfg.shape_loader)
view_sampler = instantiate(cfg.view_sampler)
shape_estimator = instantiate(cfg.shape_estimator)
shape_prior = instantiate(cfg.shape_prior)
diagnostic_refiner = instantiate(cfg.shape_refiner)
refiner_kwargs = OmegaConf.to_container(cfg.shape_refiner, resolve=True)
refiner_kwargs.pop("_target_")
shape_refiner = TracingDenseAnchorPointToPlanePRIMARefiner(**refiner_kwargs)
output_handler = instantiate(cfg.output_handler)

initialization_started = time.perf_counter()
shape_estimator.init_estimator()
shape_prior.load()
shape_prior.load_from_estimator(shape_estimator)
initialization_seconds = time.perf_counter() - initialization_started

shape_files = shape_loader.find_shape_files()
assert shape_files == [str(RAW_PATH)], shape_files
print(f"Initialized production models in {initialization_seconds:.1f} s")
print(f"Input:  {shape_files[0]}")
print(f"Output: {output_handler.out_dir}")

## 4. Load `hippo_05` and render the production 18-view camera set

In [ ]:
shape = shape_loader.pipeline(shape_files[0])
raw_mesh = shape["mesh"]
camera_views = view_sampler.sample_views()
render_started = time.perf_counter()
view_sampling = view_sampler.pipeline(shape=shape, camera_views=camera_views)
render_seconds = time.perf_counter() - render_started

assert len(camera_views) == 18
assert len(view_sampling["observations"]) == 18
print(f"Raw mesh: {len(raw_mesh.vertices):,} vertices / {len(raw_mesh.faces):,} faces")
print(f"Rendered {len(camera_views)} views in {render_seconds:.1f} s")

for view in ("xy", "yz", "zx"):
    plotter = pv.Plotter()
    plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#bdbdbd", smooth_shading=True)
    plotter.add_title(f"Raw SMAL_r {SHAPE_NAME} — {view}", font_size=10)
    set_parallel_view(plotter, view)
    plotter.show(jupyter_backend=PYVISTA_BACKEND)

In [ ]:
observation_items = list(view_sampling["observations"].items())
fig, axes = plt.subplots(3, 6, figsize=(16, 8), constrained_layout=True)
for axis, (view_key, observation) in zip(axes.flat, observation_items):
    axis.imshow(observation["rgb"])
    axis.set_title(f"az={view_key[0]:g}, el={view_key[1]:g}", fontsize=8)
    axis.axis("off")
fig.suptitle("Production camera observations for hippo_05")
plt.show()

## 5. Fresh PRIMA inference, candidate selection, and alignment

This cell does not read an existing fitted mesh. PRIMA runs on the rendered RGB observations, selects the best retained detection in each view, and aligns its SMAL prediction to the raw depth surface.

In [ ]:
frontend_started = time.perf_counter()
shape_estimation = shape_estimator.pipeline(view_sampling=view_sampling)
frontend_seconds = time.perf_counter() - frontend_started
results = shape_estimation["results"]
view_keys = list(results)

assert shape_estimation["shape"]["path"] == str(RAW_PATH)
assert len(results) >= 3, "Too few PRIMA views survived for multi-view fitting."
alignment_table = []
for view_key, result in results.items():
    confidence = result.get("confidence", {})
    alignment_table.append(
        (
            view_key,
            float(result["iou"]),
            float(confidence.get("alignment_rmse", np.nan)),
            int(confidence.get("num_inliers", len(result["alignment"]["points_tgt_ori"]))),
            float(confidence.get("inlier_ratio", np.nan)),
        )
    )

print(f"Fresh PRIMA/alignment views: {len(results)} / {len(camera_views)} in {frontend_seconds:.1f} s")
print("view             IoU       RMSE      inliers   ratio")
for row in alignment_table:
    print(f"{str(row[0]):16s} {row[1]:.4f}   {row[2]:.6f}   {row[3]:7d}   {row[4]:.4f}")

In [ ]:
panel_ids = np.linspace(0, len(view_keys) - 1, min(6, len(view_keys)), dtype=int)
fig, axes = plt.subplots(2, len(panel_ids), figsize=(3 * len(panel_ids), 6), constrained_layout=True)
for column, view_index in enumerate(panel_ids):
    view_key = view_keys[view_index]
    result = results[view_key]
    axes[0, column].imshow(view_sampling["observations"][view_key]["rgb"])
    axes[0, column].set_title(f"input {view_key}", fontsize=8)
    axes[1, column].imshow(
        rgba_overlay(view_sampling["observations"][view_key]["rgb"], result["img_rgba_smal"])
    )
    axes[1, column].set_title(f"aligned PRIMA, IoU={result['iou']:.3f}", fontsize=8)
    axes[0, column].axis("off")
    axes[1, column].axis("off")
plt.show()

## 6. Exact dense-anchor evidence, triangulation, and covariance gate

A separate production refiner instance expands the pre-optimization anchor computation. It consumes the fresh PRIMA result dictionary and uses the same canonical template, geodesic-FPS cache, projection convention, visibility tests, robust triangulation, and covariance gate as the refiner that will optimize the mesh.

In [ ]:
betas_initial, global_orient_initial, pose_initial = diagnostic_refiner.stack_smal_params(results)
canonical_vertices = diagnostic_refiner._neutral_template(
    shape_prior, betas_initial, global_orient_initial, pose_initial
)
faces = np.asarray(shape_prior.faces, dtype=np.int64)
anchor_indices = diagnostic_refiner._load_or_create_anchors(canonical_vertices, faces)
diagnostic_refiner._canonical_vertices = canonical_vertices
diagnostic_refiner._anchor_indices = anchor_indices
anchor_observations, anchor_used_views, anchor_confidence = diagnostic_refiner._triangulate_anchors(
    results, raw_mesh
)
triangulated_anchors = diagnostic_refiner._anchor_targets
final_anchor_mask = diagnostic_refiner._anchor_valid
projection_matrices = diagnostic_refiner._projection_data(results)
height = int(next(iter(results.values()))["cam_smal"]["intrinsics"]["h"])
triangulation_uv = anchor_observations.copy()
triangulation_uv[:, :, 1] = height - triangulation_uv[:, :, 1]
anchor_covariances, anchor_support = triangulation_covariances(
    projection_matrices,
    triangulated_anchors,
    triangulation_uv,
    anchor_confidence,
    confidence_threshold=diagnostic_refiner.conf_thresh,
    pixel_sigma=diagnostic_refiner.anchor_pixel_sigma,
    damping=diagnostic_refiner.covariance_damping,
    uncertainty_floor=diagnostic_refiner.covariance_floor,
)
finite_anchors = np.all(np.isfinite(triangulated_anchors), axis=1)
expanded_mask = (
    finite_anchors
    & np.all(np.isfinite(anchor_covariances), axis=(1, 2))
    & (anchor_support >= diagnostic_refiner.min_anchor_views)
)
np.testing.assert_array_equal(expanded_mask, final_anchor_mask)

print(f"Cached canonical anchors: {len(anchor_indices):,}")
print(f"Usable 2D observations:  {np.isfinite(anchor_observations).all(axis=2).sum():,}")
print(f"Finite triangulations:   {finite_anchors.sum():,}")
print(f"Finally selected:        {final_anchor_mask.sum():,}")

In [ ]:
for view in ("xy", "yz", "zx"):
    plotter = pv.Plotter()
    plotter.add_mesh(as_polydata(canonical_vertices, faces), color="#b3cde3", opacity=0.62, smooth_shading=True)
    plotter.add_points(canonical_vertices[anchor_indices], color="#d7301f", point_size=5, render_points_as_spheres=True)
    plotter.add_title(f"1,024 canonical SMAL anchors — {view}", font_size=10)
    set_parallel_view(plotter, view)
    plotter.show(jupyter_backend=PYVISTA_BACKEND)

In [ ]:
projected_candidates = []
for result in results.values():
    predicted_vertices = result["prima_output"]["pred_vertices"][0].detach().cpu().numpy()
    camera = result["cam_smal"]
    pixels, _ = project_smal_anchor_vertices(
        predicted_vertices[anchor_indices],
        camera["pred_cam_t_full"],
        camera["focal_length"],
        camera["intrinsics"]["camera_center"],
    )
    projected_candidates.append(pixels)
projected_candidates = np.stack(projected_candidates)

panel_ids = np.linspace(0, len(view_keys) - 1, min(4, len(view_keys)), dtype=int)
for view_index in panel_ids:
    view_key = view_keys[view_index]
    accepted = np.all(np.isfinite(anchor_observations[view_index]), axis=1)
    plt.figure(figsize=(5, 5))
    plt.imshow(view_sampling["observations"][view_key]["rgb"])
    plt.scatter(projected_candidates[view_index, :, 0], projected_candidates[view_index, :, 1], s=4, c="orange", alpha=0.18, label="all candidates")
    plt.scatter(anchor_observations[view_index, accepted, 0], anchor_observations[view_index, accepted, 1], s=7, c="lime", alpha=0.8, label="production-usable")
    plt.title(f"Anchor evidence {view_key}: {accepted.sum()} / 1024 usable")
    plt.xlim(0, 512)
    plt.ylim(512, 0)
    plt.axis("off")
    plt.legend(loc="lower left", fontsize=8)
    plt.show()

In [ ]:
rejected_anchors = finite_anchors & ~final_anchor_mask
plotter = pv.Plotter(shape=(1, 2), window_size=(1500, 650))
plotter.subplot(0, 0)
plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#d9d9d9", opacity=0.35, smooth_shading=True)
plotter.add_points(triangulated_anchors[finite_anchors], color="#f16913", point_size=6, render_points_as_spheres=True)
plotter.add_title(f"All finite triangulations: {finite_anchors.sum()}")
plotter.subplot(0, 1)
plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#d9d9d9", opacity=0.35, smooth_shading=True)
if np.any(rejected_anchors):
    plotter.add_points(triangulated_anchors[rejected_anchors], color="#d73027", point_size=7, render_points_as_spheres=True, label="rejected")
plotter.add_points(triangulated_anchors[final_anchor_mask], color="#1a9850", point_size=7, render_points_as_spheres=True, label="accepted")
plotter.add_legend()
plotter.add_title(f"Covariance/support gate: {final_anchor_mask.sum()} accepted")
plotter.link_views()
plotter.view_yz()
plotter.enable_parallel_projection()
plotter.show(jupyter_backend=PYVISTA_BACKEND)

## 7. Run the untouched production lifecycle and output handler

The tracing refiner now starts from the fresh `shape_estimation` dictionary. The call includes robust joint initialization, dense-anchor fitting, point-to-plane ICP with frozen correspondences refreshed every ten steps, the complete 100-step Chamfer polish, and confidence-weighted fusion.

In [ ]:
refiner_started = time.perf_counter()
estimation_refined = shape_refiner.pipeline(
    shape_estimation=shape_estimation,
    shape_prior=shape_prior,
)
refiner_seconds = time.perf_counter() - refiner_started
output_handler.pipeline(estimation_refined=estimation_refined)
articulated_mesh = estimation_refined["mesh_merged"]

np.testing.assert_allclose(
    shape_refiner._anchor_targets, triangulated_anchors, rtol=1e-7, atol=1e-8, equal_nan=True
)
np.testing.assert_array_equal(shape_refiner._anchor_valid, final_anchor_mask)
assert len(shape_refiner.icp_refresh_history) == 50
assert len(shape_refiner.icp_boundary_vertices) == 2
assert (output_handler.out_dir / "smal" / f"{SHAPE_NAME}.obj").is_file()
assert (output_handler.out_dir / "smal" / f"{SHAPE_NAME}.npy").is_file()
assert (output_handler.out_dir / "smal" / f"{SHAPE_NAME}.confidence.npz").is_file()

confidence = estimation_refined["confidence"]
optimization_summary = {
    "fresh_prima_views": len(results),
    "accepted_anchors": int(np.asarray(confidence["accepted_anchor_count"]).item()),
    "grounded_anchors": int(np.asarray(confidence["grounded_anchor_count"]).item()),
    "icp_steps": int(np.asarray(confidence["icp_steps"]).item()),
    "icp_refresh_count": int(np.asarray(confidence["icp_refresh_count"]).item()),
    "icp_handoff": str(np.asarray(confidence["icp_chamfer_handoff"]).item()),
    "icp_pre_validation_chamfer": float(np.asarray(confidence["icp_pre_validation_chamfer"]).item()),
    "icp_post_validation_chamfer": float(np.asarray(confidence["icp_post_validation_chamfer"]).item()),
    "dense_steps": int(np.asarray(confidence["dense_steps"]).item()),
    "dense_best_score": float(np.asarray(confidence["dense_best_score"]).item()),
    "frontend_seconds": float(frontend_seconds),
    "refiner_seconds": float(refiner_seconds),
}
optimization_summary

## 8. Joint, anchor-grounding, and optimization-stage geometry

In [ ]:
triangulated_joints = estimation_refined["triangulated_keypoints"]
bone_pairs = [
    pair for pair in shape_prior.get_skeleton()
    if pair[0] < len(triangulated_joints) and pair[1] < len(triangulated_joints)
]
finite_bones = [
    pair for pair in bone_pairs
    if np.isfinite(triangulated_joints[list(pair)]).all()
]
plotter = pv.Plotter()
plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#d9d9d9", opacity=0.32, smooth_shading=True)
plotter.add_points(triangulated_joints[np.isfinite(triangulated_joints).all(axis=1)], color="#2c7fb8", point_size=10, render_points_as_spheres=True)
if finite_bones:
    starts = np.stack([triangulated_joints[a] for a, _ in finite_bones])
    ends = np.stack([triangulated_joints[b] for _, b in finite_bones])
    plotter.add_mesh(line_polydata(starts, ends), color="#2c7fb8", line_width=3)
plotter.add_title("Robustly triangulated PRIMA/SMAL skeleton")
set_parallel_view(plotter, "yz")
plotter.show(jupyter_backend=PYVISTA_BACKEND)

ground_targets, ground_normals, grounded_mask, body_diagonal = shape_refiner._grounding
line_ids = np.flatnonzero(grounded_mask)
if len(line_ids) > 300:
    line_ids = line_ids[np.linspace(0, len(line_ids) - 1, 300, dtype=int)]
plotter = pv.Plotter()
plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#d9d9d9", opacity=0.35, smooth_shading=True)
plotter.add_points(shape_refiner._anchor_targets[shape_refiner._anchor_valid], color="#f16913", point_size=6, render_points_as_spheres=True, label="triangulated anchors")
if len(line_ids):
    plotter.add_mesh(line_polydata(shape_refiner._anchor_targets[line_ids], ground_targets[line_ids]), color="#756bb1", line_width=1, label="nearest raw grounding")
plotter.add_legend()
plotter.add_title("Anchor triangulations and accepted raw-surface grounding")
set_parallel_view(plotter, "yz")
plotter.show(jupyter_backend=PYVISTA_BACKEND)

In [ ]:
stage_vertices = {
    "post joint": shape_refiner.stage_vertices["post_joint"],
    "post sparse anchors": shape_refiner.stage_vertices["post_sparse"],
    "pre ICP": shape_refiner.icp_boundary_vertices[0],
    "post ICP": shape_refiner.icp_boundary_vertices[1],
    "post Chamfer": shape_refiner.stage_vertices["post_chamfer"],
}

for stage_name, vertices_by_view in stage_vertices.items():
    mean_vertices = np.mean(vertices_by_view, axis=0)
    plotter = pv.Plotter()
    plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#bdbdbd", opacity=0.30, smooth_shading=True, label="raw")
    plotter.add_mesh(as_polydata(mean_vertices, faces), color="#d7301f", opacity=0.66, smooth_shading=True, label=stage_name)
    plotter.add_legend()
    plotter.add_title(f"Mean per-view surface: {stage_name}")
    set_parallel_view(plotter, "yz")
    plotter.show(jupyter_backend=PYVISTA_BACKEND)

## 9. Point-to-plane ICP correspondences and metric trajectory

Correspondences are selected on the CPU from deterministic model vertices to one seeded 50,000-point target sample. Distance and absolute normal compatibility gates are applied per view; accepted targets and normals then remain frozen for ten gradient steps. A good scalar residual does not guarantee the correct semantic correspondence or articulation.

In [ ]:
refresh_history = np.stack(shape_refiner.icp_refresh_history)
refresh_steps = np.arange(len(refresh_history)) * cfg.shape_refiner.icp_correspondence_interval
plt.figure(figsize=(11, 5))
for view_index, view_key in enumerate(view_keys):
    plt.plot(refresh_steps, refresh_history[:, view_index], alpha=0.45, linewidth=1, label=str(view_key))
plt.plot(refresh_steps, refresh_history.mean(axis=1), color="black", linewidth=3, label="mean")
plt.xlabel("ICP step at correspondence refresh")
plt.ylabel("accepted correspondences")
plt.title("Per-view point-to-plane correspondence support")
plt.legend(ncol=4, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

print("First refresh counts:", refresh_history[0].tolist())
print("Final refresh counts:", refresh_history[-1].tolist())
print("Any normal-filter fallback:", bool(np.asarray(confidence["icp_normal_filter_fallback"]).item()))

In [ ]:
snapshot = shape_refiner.icp_latest_snapshot
view_index = int(np.argmax(snapshot["accepted_per_view"]))
valid_ids = np.flatnonzero(snapshot["valid"][view_index])
if len(valid_ids) > 350:
    valid_ids = valid_ids[np.linspace(0, len(valid_ids) - 1, 350, dtype=int)]
source_points = snapshot["source"][view_index, valid_ids]
target_points = snapshot["targets"][view_index, valid_ids]
target_normals = snapshot["normals"][view_index, valid_ids]
plane_residual = np.abs(np.sum((source_points - target_points) * target_normals, axis=1)) / body_diagonal

correspondence_lines = line_polydata(source_points, target_points)
correspondence_lines["plane residual / diagonal"] = np.tile(plane_residual, 2)
plotter = pv.Plotter()
plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#d9d9d9", opacity=0.25, smooth_shading=True)
plotter.add_points(source_points, color="#2c7fb8", point_size=5, render_points_as_spheres=True, label="model samples")
plotter.add_points(target_points, color="#31a354", point_size=5, render_points_as_spheres=True, label="target matches")
plotter.add_mesh(correspondence_lines, scalars="plane residual / diagonal", cmap="magma", line_width=2, scalar_bar_args={"title": "|n·(x-y)| / diagonal"})
plotter.add_legend()
plotter.add_title(f"Latest frozen ICP matches for view {view_keys[view_index]}")
set_parallel_view(plotter, "yz")
plotter.show(jupyter_backend=PYVISTA_BACKEND)

plt.figure(figsize=(7, 4))
plt.hist(plane_residual, bins=50, color="#756bb1")
plt.xlabel("absolute point-to-plane residual / body diagonal")
plt.ylabel("correspondences")
plt.title("Latest-refresh residuals in the displayed view")
plt.show()

In [ ]:
metric_stages = ["pre ICP", "post ICP", "post Chamfer (best fixed validation)"]
metric_values = [
    float(np.asarray(confidence["icp_pre_validation_chamfer"]).item()),
    float(np.asarray(confidence["icp_post_validation_chamfer"]).item()),
    float(np.asarray(confidence["dense_best_score"]).item()),
]
plt.figure(figsize=(8, 4))
bars = plt.bar(metric_stages, metric_values, color=["#9ecae1", "#3182bd", "#de2d26"])
plt.ylabel("fixed validation Chamfer")
plt.title("The fitting metric can improve while hippo_05 remains visually wrong")
for bar, value in zip(bars, metric_values):
    plt.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.6g}", ha="center", va="bottom")
plt.show()

## 10. Final raw-surface error, confidence-weighted fusion, and known failure comparison

In [ ]:
raw_diagonal = max(float(np.linalg.norm(np.ptp(raw_mesh.bounds, axis=0))), 1e-12)
model_to_raw = sampled_surface_gap(articulated_mesh, raw_mesh, seed=0)
raw_to_model = sampled_surface_gap(raw_mesh, articulated_mesh, seed=1)
comparison_metrics = {
    "model_to_raw_mean/body_diagonal": model_to_raw[0] / raw_diagonal,
    "raw_to_model_mean/body_diagonal": raw_to_model[0] / raw_diagonal,
    "two_way_mean/body_diagonal": 0.5 * (model_to_raw[0] + raw_to_model[0]) / raw_diagonal,
}
if CLI_REFERENCE_PATH.is_file():
    cli_reference_mesh = trimesh.load(CLI_REFERENCE_PATH, process=False)
    notebook_to_cli = sampled_surface_gap(articulated_mesh, cli_reference_mesh, seed=2)
    cli_to_notebook = sampled_surface_gap(cli_reference_mesh, articulated_mesh, seed=3)
    comparison_metrics["fresh_vs_existing_cli_two_way_mean/body_diagonal"] = (
        0.5 * (notebook_to_cli[0] + cli_to_notebook[0]) / raw_diagonal
    )

for view in ("xy", "yz", "zx"):
    plotter = pv.Plotter()
    plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#bdbdbd", opacity=0.35, smooth_shading=True, label="raw hippo_05")
    plotter.add_mesh(as_polydata(articulated_mesh.vertices, articulated_mesh.faces), color="#d7301f", opacity=0.64, smooth_shading=True, label="fresh fused result")
    plotter.add_legend()
    plotter.add_title(f"Raw vs final point-to-plane result — {view}")
    set_parallel_view(plotter, view)
    plotter.show(jupyter_backend=PYVISTA_BACKEND)

comparison_metrics

In [ ]:
_, vertex_gap, _ = closest_points_on_mesh(raw_mesh, articulated_mesh.vertices)
distance_mesh = as_polydata(articulated_mesh.vertices, articulated_mesh.faces)
distance_mesh["nearest raw gap / body diagonal"] = vertex_gap / raw_diagonal
uncertainty_mesh = as_polydata(articulated_mesh.vertices, articulated_mesh.faces)
uncertainty_mesh["fusion uncertainty"] = np.asarray(confidence["vertex_uncertainty"])
support_mesh = as_polydata(articulated_mesh.vertices, articulated_mesh.faces)
support_mesh["visibility support"] = np.asarray(confidence["vertex_visibility_support"])

for title, mesh, scalar, cmap in (
    ("Final distance to raw", distance_mesh, "nearest raw gap / body diagonal", "magma"),
    ("Fusion uncertainty", uncertainty_mesh, "fusion uncertainty", "inferno"),
    ("Weighted visibility support", support_mesh, "visibility support", "viridis"),
):
    plotter = pv.Plotter()
    plotter.add_mesh(mesh, scalars=scalar, cmap=cmap, smooth_shading=True, scalar_bar_args={"title": scalar})
    plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="white", opacity=0.10, style="wireframe")
    plotter.add_title(title)
    set_parallel_view(plotter, "yz")
    plotter.show(jupyter_backend=PYVISTA_BACKEND)

fallback = np.asarray(confidence["vertex_visibility_fallback"], dtype=bool)
print(f"Fusion fallback vertices: {fallback.sum()} / {len(fallback)}")

In [ ]:
residual_names = np.asarray(confidence["residual_names"]).tolist()
residual_terms = np.asarray(confidence["view_residual_terms"])
view_weights = np.asarray(confidence["view_weights"])
x = np.arange(len(view_keys))
fig, axes = plt.subplots(2, 1, figsize=(13, 7), constrained_layout=True, sharex=True)
bottom = np.zeros(len(view_keys))
for term_index, name in enumerate(residual_names):
    axes[0].bar(x, residual_terms[:, term_index], bottom=bottom, label=name)
    bottom += residual_terms[:, term_index]
axes[0].set_ylabel("diagnostic residual terms")
axes[0].legend(ncol=len(residual_names), fontsize=8)
axes[1].bar(x, view_weights, color="#3182bd")
axes[1].set_ylabel("fusion weight")
axes[1].set_xticks(x, [str(key) for key in view_keys], rotation=60, ha="right")
fig.suptitle("Per-view residual diagnostics and confidence-weighted fusion")
plt.show()

## Visual correctness checklist

- The composed config must retain all 18 views, 1,000 sparse-anchor steps, 500 ICP steps, 50 correspondence refreshes, and the full 100-step Chamfer polish.
- PRIMA overlays should expose any incorrect silhouette or part alignment before optimization; these errors can seed coherent but semantically wrong anchors.
- Orange candidate projections and green usable anchors must come from the fresh `prima_output`, not a saved fitted mesh.
- The expanded anchor mask must exactly equal the production refiner mask. Broadly coherent but misplaced anchors around the torso or limbs are more important than isolated scan debris.
- ICP correspondence counts can remain high while matches cross semantic parts. Inspect the correspondence lines and residual field rather than treating high support as correctness.
- Compare post-joint, post-sparse, pre-ICP, post-ICP, and post-Chamfer meshes. The first stage where a limb folds, swaps sides, or collapses onto the torso identifies the responsible objective.
- A lower fixed-validation Chamfer is geometric evidence, not proof of correct articulation or correspondence. `hippo_05` is specifically included to make this distinction visible.
- Fusion uncertainty and fallback reveal disagreement among final per-view hypotheses, but low uncertainty can still occur when every view converges to the same wrong configuration.
- The optional fresh-versus-existing-CLI comparison should be small, but exact equality is not required because optimization surface samples are not globally seeded.